In [1]:
import duckdb
import os
from pathlib import Path

from balsa.envs.envs import ParseSqlToNode
from balsa.util import plans_lib
import balsa.util.duck_db as duck_db
import pg_executor.pg_executor as pg

/home/wangshuhong/miniconda3/envs/balsa/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dsn = os.path.join(str(Path.home()), 'duckdb', 'imdb.db')

In [16]:
def generare_and_run_sql(path):
	
	original_sql_str = ''

	with open(path, 'r') as f:
		original_sql_str = f.read()

	node = ParseSqlToNode(path)
	sql = node.generate_duckdb_sql()
	print('SQL for DuckDB: {}'.format(sql))

	duckdb_res = duck_db.Execute(sql, use_optimizer=True).result
	duckdb_original_res = duck_db.Execute(original_sql_str, use_optimizer=True).result
	# postgres_res = ''
	# with pg.Cursor() as cursor:
	# 	postgres_res = pg.Execute(original_sql_str, cursor=cursor).result

	if duckdb_res != duckdb_original_res:
		print('Result does not match!')
		print('Generated SQL result: {}'.format(duckdb_res))
		print('Original SQL result: {}'.format(duckdb_original_res))
		return False
	
	# with duckdb.connect(dsn) as con:
	# 	con.execute("SET disabled_optimizers = 'join_order,build_side_probe_side'")
	# 	res = con.sql(sql).explain()
	# 	print(res)

In [17]:
generare_and_run_sql(os.path.join('queries', 'join-order-benchmark', '9c.sql'))

SQL for DuckDB: SELECT min(an.name),min(n.name),min(chn.name),min(t.title) FROM ((((((((aka_name AS an CROSS JOIN name AS n) CROSS JOIN cast_info AS ci) CROSS JOIN role_type AS rt) CROSS JOIN char_name AS chn) CROSS JOIN movie_companies AS mc) CROSS JOIN company_name AS cn) CROSS JOIN title AS t)) WHERE an.person_id = ci.person_id AND an.person_id = n.id AND chn.id = ci.person_role_id AND ci.movie_id = mc.movie_id AND ci.movie_id = t.id AND ci.person_id = n.id AND ci.role_id = rt.id AND cn.id = mc.company_id AND mc.movie_id = t.id AND ((n.name LIKE '%An%') AND ((n.gender) = 'f')) AND ci.note IN ('(voice)','"(voice: Japanese version)"','"(voice) (uncredited)"','"(voice: English version)"') AND ((rt.role) = 'actress') AND ((cn.country_code) = '[us]');
Result does not match!
Generated SQL result: [("'Annette'", 'Alborg, Ana Esther', "80's Robin", '(1975-01-20)')]
Original SQL result: [("'Annette'", '2nd Balladeer', 'Alborg, Ana Esther', '(1975-01-20)')]


False

In [8]:
def run_sql(query: str):
	with duckdb.connect(dsn) as con:
		# con.execute("SET memory_limit = '32GB'")
		# con.execute("SET disabled_optimizers = 'join_order,build_side_probe_side'")
		res = con.sql(query)
		print(res)

In [13]:
sql = '''
	SELECT 
    min(an.name),
    min(n.name),
    min(chn.name),
    min(t.title) 
FROM 
((((((((aka_name AS an CROSS JOIN name AS n) CROSS JOIN cast_info AS ci) CROSS JOIN role_type AS rt) CROSS JOIN char_name AS chn) CROSS JOIN movie_companies AS mc) CROSS JOIN company_name AS cn) CROSS JOIN title AS t))
WHERE ci.note IN ('(voice)',
    '(voice: Japanese version)',
    '(voice) (uncredited)',
    '(voice: English version)') 
AND cn.country_code = '[us]'
AND an.person_id = ci.person_id 
AND an.person_id = n.id 
AND chn.id = ci.person_role_id 
AND ci.movie_id = mc.movie_id 
AND ci.movie_id = t.id 
AND ci.person_id = n.id 
AND ci.role_id = rt.id 
AND cn.id = mc.company_id 
AND mc.movie_id = t.id 
AND n.name LIKE '%An%'
AND n.gender = 'f'
AND rt.role = 'actress';
'''

print(run_sql(sql))

┌────────────────┬────────────────────┬─────────────────┬──────────────┐
│ min(an."name") │   min(n."name")    │ min(chn."name") │ min(t.title) │
│    varchar     │      varchar       │     varchar     │   varchar    │
├────────────────┼────────────────────┼─────────────────┼──────────────┤
│ 'Annette'      │ Alborg, Ana Esther │ 2nd Balladeer   │ (1975-01-20) │
└────────────────┴────────────────────┴─────────────────┴──────────────┘

None


In [8]:
def explain_sql(query: str):
	with duckdb.connect(dsn) as con:
		con.execute("SET memory_limit = '32GB'")
		con.execute("SET disabled_optimizers = 'join_order,build_side_probe_side'")
		# con.execute("PRAGMA explain_output = 'optimized_only'")
		print(con.sql(query).explain())
		# print(con.sql(query))

In [6]:
def explain_optimized_sql(query: str):
	with duckdb.connect(dsn) as con:
		con.execute("SET memory_limit = '32GB'")
		print(con.sql(query))

In [26]:
### Default
default_query = '''
	SELECT MIN(mc.note) AS production_note,
       MIN(t.title) AS movie_title,
       MIN(t.production_year) AS movie_year
	FROM company_type AS ct,
		info_type AS it,
		movie_companies AS mc,
		movie_info_idx AS mi_idx,
		title AS t
	WHERE ct.kind = 'production companies'
	AND it.info = 'top 250 rank'
	AND mc.note NOT LIKE '%(as Metro-Goldwyn-Mayer Pictures)%'
	AND (mc.note LIKE '%(co-production)%'
		OR mc.note LIKE '%(presents)%')
	AND ct.id = mc.company_type_id
	AND t.id = mc.movie_id
	AND t.id = mi_idx.movie_id
	AND mc.movie_id = mi_idx.movie_id
	AND it.id = mi_idx.info_type_id;
'''
explain_optimized_sql(default_query)

┌────────────────────────────────────────────────────┬────────────────────┬────────────┐
│                  production_note                   │    movie_title     │ movie_year │
│                      varchar                       │      varchar       │   int32    │
├────────────────────────────────────────────────────┼────────────────────┼────────────┤
│ (A Warner Bros.-First National Picture) (presents) │ A Clockwork Orange │       1934 │
└────────────────────────────────────────────────────┴────────────────────┴────────────┘



In [9]:
### Balsa
balsa_query = '''
	SELECT min(mc.note),min(t.title),min(t.production_year) FROM (((movie_companies AS mc CROSS JOIN company_type AS ct) CROSS JOIN (title AS t CROSS JOIN (movie_info_idx AS mi_idx CROSS JOIN info_type AS it)))) WHERE ct.id = mc.company_type_id AND it.id = mi_idx.info_type_id AND mc.movie_id = mi_idx.movie_id AND mc.movie_id = t.id AND mi_idx.movie_id = t.id AND ((mc.note NOT LIKE '%(as Metro-Goldwyn-Mayer Pictures)%') AND (mc.note LIKE '%(co-production)%')) AND ((ct.kind) = 'production companies') AND (t.production_year > 2010) AND ((it.info) = 'top 250 rank');
'''
explain_sql(balsa_query)
